# Mini-project — Embedding similarity explorer

## Mission

Build a tiny search tool that ranks words by cosine similarity. This combines array creation,
shapes, normalization, matrix multiplication, indexing, sorting, and functions.

The vectors below are handmade teaching data, not real pretrained embeddings.


In [1]:
import numpy as np


## Step 1 — Store words and vectors


In [2]:
words = np.array([
    "زبان", "فارسی", "متن", "پردازش", "رایانه", "مدرسه", "آموزش"
])

embeddings = np.array([
    [0.90, 0.80, 0.10, 0.20],
    [0.88, 0.78, 0.12, 0.18],
    [0.70, 0.35, 0.55, 0.25],
    [0.62, 0.25, 0.70, 0.35],
    [0.15, 0.10, 0.90, 0.40],
    [0.45, 0.75, 0.20, 0.65],
    [0.50, 0.80, 0.25, 0.70],
], dtype=np.float32)

print(words.shape)
print(embeddings.shape)


(7,)
(7, 4)


## Step 2 — Validate the data

Good projects check assumptions early.


In [3]:
assert embeddings.ndim == 2
assert len(words) == embeddings.shape[0]
assert np.all(np.linalg.norm(embeddings, axis=1) > 0)
print("Validation passed.")


Validation passed.


## Step 3 — Normalize the embeddings


In [4]:
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
normalized = embeddings / norms

print(np.round(np.linalg.norm(normalized, axis=1), 6))


[1. 1. 1. 1. 1. 1. 1.]


## Step 4 — Write a search function

The query vector is one row. Multiplying all normalized rows by that vector computes all cosine
similarities at once.


In [5]:
def most_similar(query_word, top_k=3):
    matches = np.where(words == query_word)[0]
    if len(matches) == 0:
        raise ValueError(f"Unknown word: {query_word}")

    query_index = int(matches[0])
    similarities = normalized @ normalized[query_index]
    ranking = np.argsort(similarities)[::-1]
    ranking = ranking[ranking != query_index][:top_k]

    return [
        (str(words[index]), float(similarities[index]))
        for index in ranking
    ]


## Step 5 — Try the tool


In [6]:
for word, score in most_similar("زبان", top_k=3):
    print(f"{word}: {score:.3f}")


فارسی: 1.000
آموزش: 0.853
مدرسه: 0.851


In [7]:
for word, score in most_similar("آموزش", top_k=3):
    print(f"{word}: {score:.3f}")


مدرسه: 1.000
زبان: 0.853
فارسی: 0.850


## Step 6 — Build a full similarity table


In [8]:
similarity_table = normalized @ normalized.T
print(np.round(similarity_table, 2))


[[1.   1.   0.84 0.71 0.31 0.85 0.85]
 [1.   1.   0.85 0.72 0.33 0.85 0.85]
 [0.84 0.85 1.   0.98 0.74 0.78 0.79]
 [0.71 0.72 0.98 1.   0.86 0.73 0.75]
 [0.31 0.33 0.74 0.86 1.   0.53 0.55]
 [0.85 0.85 0.78 0.73 0.53 1.   1.  ]
 [0.85 0.85 0.79 0.75 0.55 1.   1.  ]]


## Step 7 — Test expected behavior


In [9]:
assert similarity_table.shape == (len(words), len(words))
assert np.allclose(np.diag(similarity_table), 1.0)
assert np.allclose(similarity_table, similarity_table.T)
print("All tests passed.")


All tests passed.


## Your extensions

1. Add more words and vectors.
2. Add a function that accepts a new vector rather than a known word.
3. Return the results as a pandas DataFrame after you study pandas.
4. Replace the handmade vectors with pretrained Persian embeddings later.
5. Add tests for unknown words and invalid `top_k` values.
